# 03 — Model Evaluation
Load the best saved checkpoint and evaluate it on the test set.
Produces:
- Classification report
- Confusion matrix heatmap
- Per-class performance summary

In [ ]:
import sys
sys.path.insert(0, '..')

import os
import torch
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

from utils.config import load_config
from utils.helpers import set_seed, get_device
from data.dataset_loader import build_dataloaders, CLASS_NAMES
from data.preprocessing import get_val_transform
from models.vl_jepa_model import VLJEPAModel
from evaluation.metrics import compute_metrics
from utils.visualization import plot_confusion_matrix

print('Imports OK')

In [ ]:
# ── Config & device ──────────────────────────────────────────────
cfg    = load_config('../configs')
device = get_device(cfg['device'])
set_seed(cfg['project']['seed'])
print(f'Device: {device}')

In [ ]:
# ── Build test loader ────────────────────────────────────────────
data_root = os.path.join('..', cfg['paths']['data_root'])
val_tf    = get_val_transform(cfg['dataset']['image_size'])

_, _, test_loader, vocab_size = build_dataloaders(
    train_dir       = os.path.join(data_root, 'train'),
    test_dir        = os.path.join(data_root, 'test'),
    train_transform = val_tf,
    val_transform   = val_tf,
    batch_size      = cfg['training']['batch_size'],
    num_workers     = 0,
    max_seq_len     = cfg['text_encoder']['max_seq_len'],
)

print(f'Test set size: {len(test_loader.dataset)}')

In [ ]:
# ── Load checkpoint ───────────────────────────────────────────────
ckpt_path = os.path.join('..', cfg['paths']['checkpoint_dir'], cfg['training']['checkpoint_name'])
print(f'Loading checkpoint: {ckpt_path}')

model = VLJEPAModel(
    vocab_size     = vocab_size,
    embedding_dim  = cfg['image_encoder']['embedding_dim'],
    projection_dim = cfg['vl_jepa']['projection_dim'],
    num_classes    = cfg['vl_jepa']['num_classes'],
    dropout        = cfg['vl_jepa']['dropout'],
    use_text       = cfg['vl_jepa']['use_text_branch'],
)

ckpt = torch.load(ckpt_path, map_location=device)
model.load_state_dict(ckpt['model_state'])
model = model.to(device)
model.eval()

print(f'Checkpoint epoch: {ckpt["epoch"]} | Val Loss: {ckpt["val_loss"]:.4f}')

In [ ]:
# ── Run inference ─────────────────────────────────────────────────
all_preds, all_labels = [], []

with torch.no_grad():
    for images, tokens, labels in tqdm(test_loader, desc='Inference'):
        images = images.to(device)
        tokens = tokens.to(device)
        out    = model(images, tokens)
        preds  = out['logits'].argmax(dim=1).cpu().tolist()
        all_preds.extend(preds)
        all_labels.extend(labels.tolist())

print(f'Inference complete — {len(all_preds)} samples')

In [ ]:
# ── Metrics report ────────────────────────────────────────────────
metrics = compute_metrics(all_labels, all_preds, class_names=CLASS_NAMES, verbose=True)

In [ ]:
# ── Confusion matrix plot ─────────────────────────────────────────
cm      = metrics['confusion_matrix']
cm_norm = cm.astype(float) / (cm.sum(axis=1, keepdims=True) + 1e-8)

fig, ax = plt.subplots(figsize=(7, 6))
im      = ax.imshow(cm_norm, cmap='Blues')
plt.colorbar(im, ax=ax)

n = cm.shape[0]
ax.set_xticks(range(n)); ax.set_yticks(range(n))
ax.set_xticklabels(CLASS_NAMES, rotation=30, ha='right', fontsize=9)
ax.set_yticklabels(CLASS_NAMES, fontsize=9)
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_title('Confusion Matrix', fontsize=13, fontweight='bold')

thresh = cm_norm.max() / 2.0
for i in range(n):
    for j in range(n):
        color = 'white' if cm_norm[i, j] > thresh else 'black'
        ax.text(j, i, f'{cm[i,j]}\n({cm_norm[i,j]:.1%})',
                ha='center', va='center', fontsize=8, color=color)

plt.tight_layout()
plt.show()

print(f'\nTest Accuracy : {metrics["accuracy"]:.4f}')
print(f'Macro F1      : {metrics["f1"]:.4f}')